### Orders Data

In [1]:
import pandas as pd

df_orders = pd.read_csv("olist_orders_dataset.csv")

print("Shape:", df_orders.shape)
print("\nColumns:\n", df_orders.columns)
print("\nInfo:")
print(df_orders.info())

print("\nMissing Values:\n", df_orders.isnull().sum())

print("\nSample Data:")
df_orders.head(3)

Shape: (99441, 8)

Columns:
 Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [2]:
# Order Status Distribution
print(df_orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

print(df_orders.dtypes)

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [4]:
print("Duplicate order_id:", df_orders['order_id'].duplicated().sum())

Duplicate order_id: 0


In [5]:
# Feature Eng.
df_orders['order_month'] = df_orders['order_purchase_timestamp'].dt.to_period('M')

df_orders['delivery_time_days'] = (
    df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']
).dt.days

In [6]:
df_delivered = df_orders[df_orders['order_status']=="delivered"].copy()

In [7]:
# Check Data Quality Issues
print("Duplicate order_id:",
      df_orders['order_id'].duplicated().sum())

print("Duplicate rows:",
      df_orders.duplicated().sum())

Duplicate order_id: 0
Duplicate rows: 0


In [8]:
# Check date inconsistencies
print(
    "Delivered before purchase:",
    (df_orders['order_delivered_customer_date'] 
     < df_orders['order_purchase_timestamp']).sum()
)

Delivered before purchase: 0


In [9]:
# Check delivery time statistics
df_orders['delivery_time_days'] = (
    df_orders['order_delivered_customer_date']
    - df_orders['order_purchase_timestamp']
).dt.days
df_orders['delivery_time_days'].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_time_days, dtype: float64

In [10]:
# missing values 
df_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
order_month                         0
delivery_time_days               2965
dtype: int64

In [11]:
df_orders[
    df_orders['delivery_time_days'] == 209
][[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'delivery_time_days'
]]

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days
19590,ca07593549f1816d26a572e06dc1eab6,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,209.0


In [12]:
# Check for more extreme deliveries
df_orders['delivery_time_days'].describe(percentiles=[0.90,0.95,0.99])

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
50%         10.000000
90%         23.000000
95%         29.000000
99%         46.000000
max        209.000000
Name: delivery_time_days, dtype: float64

In [13]:
# Estimated Delivery Performance
df_orders['delivery_delay_days'] = (
    df_orders['order_delivered_customer_date']
    - df_orders['order_estimated_delivery_date']
).dt.days

In [14]:
# Delivery delay analysis
df_orders['delivery_delay_days'].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

In [15]:
# delivery_status - Early/ Late
def delivery_status(x):
    if x <= 0:
        return "Early"
    else:
        return "Late"

df_orders['delivery_status'] = (
    df_orders['delivery_delay_days']
    .apply(delivery_status)
)

In [16]:
df_orders['delivery_status'].value_counts()

delivery_status
Early    89941
Late      9500
Name: count, dtype: int64

Around 90% of delivered orders arrived before the estimated delivery date, while approximately 10% experienced delays.

In [17]:
# Check for impossible negative delivery (delivered before purchased)
df_orders[
    df_orders['delivery_time_days'] < 0
][[
    'order_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'delivery_time_days'
]]

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_time_days


In [18]:
df_orders.to_csv("clean_orders.csv", index=False)

### Customers Data

In [19]:
df_customers = pd.read_csv("olist_customers_dataset.csv")

In [20]:
print("Shape:", df_customers.shape)

print("\nColumns:")
print(df_customers.columns)

print("\nInfo:")
print(df_customers.info())

print("\nMissing Values:")
print(df_customers.isnull().sum())

print("\nDuplicates:")
print(df_customers.duplicated().sum())

print("\nHead:")
df_customers.head(3)

Shape: (99441, 5)

Columns:
Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None

Missing Values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicates:
0

Head:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [21]:
print("Unique customer_id:",
      df_customers['customer_id'].nunique())

print("Unique customer_unique_id:",
      df_customers['customer_unique_id'].nunique())

Unique customer_id: 99441
Unique customer_unique_id: 96096


In [22]:
df_customers['customer_state'].value_counts().head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

In [23]:
(df_customers['customer_state'].value_counts(normalize=True)*100).head(5)

customer_state
SP    41.980672
RJ    12.924247
MG    11.700405
RS     5.496727
PR     5.073360
Name: proportion, dtype: float64

The customer base is highly concentrated in São Paulo, indicating a strong market presence in this region

In [24]:
print(
    "Unique cities:",
    df_customers['customer_city'].nunique()
)

Unique cities: 4119


In [25]:
df_customers['customer_zip_code_prefix'].nunique()

14994

In [26]:
# Drop customer_zip_code_prefix
df_customers = df_customers.drop(columns=['customer_zip_code_prefix'])

In [27]:
# Repeat customers
customer_counts = (
    df_customers['customer_unique_id']
    .value_counts()
)

customer_counts.head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
1b6c7548a2a1f9037c1fd3ddfed95f33     7
12f5d6e1cbf93dafd9dcc19095df0b3d     6
dc813062e0fc23409cd255f7f53c7074     6
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
de34b16117594161a6a89c50b289d35a     6
63cfc61cee11cbe306bff5857d00bfe4     6
Name: count, dtype: int64

In [28]:
customer_order_counts = (
    df_customers['customer_unique_id']
    .value_counts()
)

repeat_customers = (
    customer_order_counts > 1
).sum()

total_customers = (
    customer_order_counts.shape[0]
)

print("Total unique customers:", total_customers)
print("Repeat customers:", repeat_customers)
print(
    "Repeat customer percentage:",
    round((repeat_customers/total_customers)*100,2)
)

Total unique customers: 96096
Repeat customers: 2997
Repeat customer percentage: 3.12


96.88% customers purchased only once. Only 3.12% customers came back

In [29]:
# check whether there are inconsistencies in city names
df_customers['customer_city'].value_counts().head(20)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
niteroi                    849
santo andre                797
osasco                     746
santos                     713
goiania                    692
sao jose dos campos        691
fortaleza                  654
sorocaba                   633
recife                     613
florianopolis              570
Name: count, dtype: int64

In [30]:
df_customers.to_csv(
    "clean_customers.csv",
    index=False
)

print(df_customers.shape)

(99441, 4)


### Payments Data

In [31]:
import pandas as pd
df_payments = pd.read_csv("olist_order_payments_dataset.csv")

print("Shape:", df_payments.shape)
print("\nColumns:")
print(df_payments.columns)

print("\nInfo:")
print(df_payments.info())

print("\nMissing Values:")
print(df_payments.isnull().sum())

print("\nDuplicates:")
print(df_payments.duplicated().sum())

print("\nHead:")
df_payments.head(3)

Shape: (103886, 5)

Columns:
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB
None

Missing Values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Duplicates:
0

Head:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [32]:
payment_order_counts = (
    df_payments
    .groupby('order_id')
    .size()
)

print(payment_order_counts.describe())

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64


In [33]:
df_payments[
    df_payments.groupby('order_id')['order_id'].transform('count') == 29
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
32519,fa65dad1b0e818e3ccc5cb0e39231352,11,voucher,1,4.03


In [34]:
# Check payment methods
df_payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [35]:
df_payments[
    df_payments['payment_type']=="not_defined"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [36]:
df_payments['payment_type'] = (
    df_payments['payment_type']
    .replace('not_defined','unknown')
)

In [37]:
# Check payment value distribution
df_payments['payment_value'].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [38]:
print(
    "Zero payment values:",
    (df_payments['payment_value'] == 0).sum()
)

print(
    "Negative payment values:",
    (df_payments['payment_value'] < 0).sum()
)

Zero payment values: 9
Negative payment values: 0


In [39]:
df_payments[
    df_payments['payment_value']==0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,unknown,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,unknown,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,unknown,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [40]:
df_orders = pd.read_csv("clean_orders.csv")
missing_orders = set(df_payments['order_id']) - set(df_orders['order_id'])

print("Payment orders missing from orders table:", len(missing_orders))

Payment orders missing from orders table: 0


In [41]:
# create an order-level payment table.
payment_summary = (
    df_payments
    .groupby('order_id')
    .agg(
        total_payment=('payment_value','sum'),
        payment_count=('payment_sequential','count'),
        payment_method_count=('payment_type','nunique'),
        max_installments=('payment_installments','max')
    )
    .reset_index()
)

payment_summary.head()

,order_id,total_payment,payment_count,payment_method_count,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3


In [42]:
df_payments.to_csv("clean_df_payments.csv",index=False)

print(df_payments.shape)

(103886, 5)


In [43]:
payment_summary.to_csv("payment_summary.csv", index=False)

Why did we create a separate payment_summary table?

Because the payments dataset contained multiple rows per order, directly summing payment_value would overcount revenue. I aggregated payments at the order level to create a clean, analysis-ready dataset.

### Order_items Table

In [44]:
df_items = pd.read_csv("olist_order_items_dataset.csv")

print("Shape:", df_items.shape)

print("\nColumns:")
print(df_items.columns)

print("\nInfo:")
print(df_items.info())

print("\nMissing Values:")
print(df_items.isnull().sum())

print("\nDuplicates:")
print(df_items.duplicated().sum())

print("\nHead:")
df_items.head(3)

Shape: (112650, 7)

Columns:
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB
None

Missing Values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value      

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


In [45]:
# Convert Shipping date column from object to datetime
df_items['shipping_limit_date'] = pd.to_datetime(df_items['shipping_limit_date'])

In [46]:
# Total value
df_items['total_item_value'] = (
    df_items['price'] + df_items['freight_value']
)

In [47]:
 # Does item-level revenue match payment-level revenue?
order_item_total = (
    df_items
    .groupby('order_id')['total_item_value']
    .sum()
    .reset_index(name='item_total')
)

comparison = order_item_total.merge(
    payment_summary,
    on='order_id',
    how='inner'
)

comparison['difference'] = (
    comparison['total_payment'] - comparison['item_total']
)

comparison['difference'].describe()

count    98665.000000
mean         0.029092
std          1.129221
min        -51.620000
25%          0.000000
50%          0.000000
75%          0.000000
max        182.810000
Name: difference, dtype: float64

I validated revenue consistency by comparing item-level totals with payment-level totals. Most orders matched exactly, with small deviations likely due to discounts, vouchers, and rounding effects.

In [48]:
# Total items per order
items_per_order = (
    df_items
    .groupby('order_id')
    .agg(
        total_items=('order_item_id','count'),
        unique_products=('product_id','nunique'),
        total_price=('price','sum'),
        total_freight=('freight_value','sum')
    )
    .reset_index()
)

In [49]:
df_items['total_item_value'] = df_items['price'] + df_items['freight_value']

In [50]:
df_items.to_csv("clean_order_items.csv", index=False)

### Products Table

In [51]:
df_products = pd.read_csv("olist_products_dataset.csv")
print("Shape:", df_products.shape)

print("\nColumns:")
print(df_products.columns)

print("\nInfo:")
print(df_products.info())

print("\nMissing Values:")
print(df_products.isnull().sum())

print("\nDuplicates:")
print(df_products.duplicated().sum())

print("\nHead:")
df_products.head(3)

Shape: (32951, 9)

Columns:
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


In [52]:
df_products['product_category_name'] = (
    df_products['product_category_name']
    .fillna('unknown')
)

In [53]:
cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]

df_products[cols] = df_products[cols].fillna(0)

In [54]:
physical_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

df_products[physical_cols] = df_products[physical_cols].fillna(
    df_products[physical_cols].median()
)

In [55]:
df_translation = pd.read_csv("product_category_name_translation.csv")

In [56]:
df_products = df_products.merge(
    df_translation,
    on='product_category_name',
    how='left'
)

df_products.rename(columns={
    'product_category_name_english': 'category_en'
}, inplace=True)

In [57]:
# Are all products in order_items present here?
missing_products = set(df_items['product_id']) - set(df_products['product_id'])

print("Missing products:", len(missing_products))

Missing products: 0


In [58]:
df_products.to_csv("clean_products.csv", index=False)

### Sellers Table

In [59]:
df_sellers = pd.read_csv("olist_sellers_dataset.csv")
print("Shape:", df_sellers.shape)

print("\nColumns:")
print(df_sellers.columns)

print("\nInfo:")
print(df_sellers.info())

print("\nMissing Values:")
print(df_sellers.isnull().sum())

print("\nDuplicates:")
print(df_sellers.duplicated().sum())

print("\nHead:")
df_sellers.head(3)

Shape: (3095, 4)

Columns:
Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB
None

Missing Values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Duplicates:
0

Head:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


In [60]:
# Standardize text columns
df_sellers['seller_city'] = df_sellers['seller_city'].str.title()

In [61]:
# Are all sellers in order_items present here?
missing_sellers = set(df_items['seller_id']) - set(df_sellers['seller_id'])

print("Missing sellers:", len(missing_sellers))

Missing sellers: 0


In [62]:
df_sellers.to_csv("clean_sellers.csv", index=False)

### Reviews Table

In [63]:
df_reviews = pd.read_csv("olist_order_reviews_dataset.csv")

In [64]:
print("Shape:", df_reviews.shape)

print("\nColumns:")
print(df_reviews.columns)

print("\nInfo:")
print(df_reviews.info())

print("\nMissing Values:")
print(df_reviews.isnull().sum())

print("\nDuplicates:")
print(df_reviews.duplicated().sum())

print("\nHead:")
df_reviews.head()

Shape: (99224, 7)

Columns:
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB
None

Missing Values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title   

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [65]:
# Drop columns
df_reviews = df_reviews.drop(
    columns=['review_comment_title', 'review_comment_message']
)

In [66]:
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'])
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'])

In [67]:
# Validate relationship with orders
missing_reviews = set(df_reviews['order_id']) - set(df_orders['order_id'])

print("Reviews with missing orders:", len(missing_reviews))

Reviews with missing orders: 0


In [68]:
# Feature Eng.
df_reviews['review_label'] = df_reviews['review_score'].apply(
    lambda x: 'positive' if x >= 4 else ('neutral' if x == 3 else 'negative')
)

In [69]:
df_reviews.to_csv("clean_reviews.csv", index=False)